## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails for names in message and email tool guardrails implementation

In [ ]:
import sys

from dotenv import load_dotenv
import os
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from agents.tool_guardrails import tool_input_guardrail, ToolGuardrailFunctionOutput, ToolInputGuardrailData
from typing import Dict, Optional
from pydantic import BaseModel, Field, field_validator



In [ ]:
load_dotenv(override=True)


In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")   



In [ ]:
instructions1 = "You are an expert, professional sales agent working for \
Fun-EduTech a SaaS company that provides a comprehensive Learning Management System (LMS) for schools. \
Your goal is to write serious, authoritative, and consultative cold emails to school administrators and principals. Your tone should be deeply respectful, polished, and focused on educational excellence, data security, and operational efficiency."

instructions2 = "You are a witty, warm, and highly engaging sales agent working for Fun-EduTech a SaaS company that provides a comprehensive Learning Management System (LMS) for schools.\
Your goal is to write lighthearted, empathetic cold emails to busy school principals and teachers who are overwhelmed by admin work. Use clever, relatable school-life humor and a refreshing, \
conversational tone that stands out in a crowded inbox to maximize response rates."

instructions3 = "You are a hyper-efficient, direct sales agent working for Fun-EduTech a SaaS company that provides a comprehensive Learning Management System (LMS) for schools. \
Your goal is to write ultra-brief, zero-fluff cold emails to school leadership. Cut out all introductory pleasantries and filler words. Focus purely on the immediate value proposition and a low-friction call to action,\
keeping the entire email readable in under 20 seconds."


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"


In [ ]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)


In [ ]:
# Separate quota from gemini-2.5-flash; alternatives: gemini-2.0-flash, gemini-2.5-flash-lite
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.5-flash-lite", openai_client=gemini_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

In [ ]:
sales_agent1 = Agent(name = "Gemini Sales Agent",
instructions=instructions1,
model= gemini_model
)

sales_agent2 = Agent(name = "OpenAI Sales Agent",
instructions=instructions2,
model= 'gpt-5-nano'
)

sales_agent3  = Agent(name="Llama3.3 Sales Agent",
instructions=instructions3,model=llama3_3_model)


In [ ]:
description = "Write a cold sales email"
tool1 = sales_agent1.as_tool(tool_name="sales_agent1",tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2",tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3",tool_description=description)

In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import re


In [ ]:
SENDER_EMAIL = "" # sender email
RECIPIENT_EMAIL = ""  #receiver email


In [ ]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [ ]:
# Validate sender and recipient addresses before send

class EmailValidationSchema(BaseModel):
    sender_email: str = Field(description="The sender's email address.")
    recipient_email: str = Field(description="The recipient's email address.")

    @field_validator("sender_email", "recipient_email")
    @classmethod
    def validate_email_format(cls, value: str) -> str:
        email_regex = r"^[\w\.-]+@[\w\.-]+\.\w+$"
        if not re.match(email_regex, value):
            raise ValueError(f"'{value}' is not a syntactically valid email address.")
        return value

In [ ]:
@tool_input_guardrail
async def email_validation_guardrail(data: ToolInputGuardrailData) -> ToolGuardrailFunctionOutput:
    """Validate configured sender and recipient emails before send_html_email runs."""
    try:
        EmailValidationSchema(
            sender_email=SENDER_EMAIL,
            recipient_email=RECIPIENT_EMAIL,
        )
        return ToolGuardrailFunctionOutput.allow()
    except Exception as validation_error:
        return ToolGuardrailFunctionOutput.raise_exception(
            output_info={"reason": f"Invalid email configuration: {validation_error}"}
        )

In [ ]:
@function_tool(tool_input_guardrails=[email_validation_guardrail])
async def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Sends out an email with the given subject and HTML body to the validated prospect."""
    password = ''   #your app password
    
    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"] = SENDER_EMAIL
    msg["To"] = RECIPIENT_EMAIL
    msg.attach(MIMEText(html_body, "html"))

    try:
        with smtplib.SMTP("smtp.gmail.com", 587) as connection:
            connection.starttls()
            connection.login(user=SENDER_EMAIL, password=password)
            connection.sendmail(SENDER_EMAIL, RECIPIENT_EMAIL, msg.as_string())
        return {"status": "success"}
    
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
# ==========================================
# 4. AGENT ARRANGEMENT SETUP
# ==========================================
email_tools = [subject_tool, html_tool, send_html_email]

email_instructions = (
    "You are an email formatter and sender. You receive the body of an email to be sent. "
    "You first use the subject_writer tool to write a subject for the email, then use the "
    "html_converter tool to convert the body to HTML. "
    "Finally, you use the send_html_email tool to send the email with the subject and HTML body."
)

emailer_agent = Agent(
    name="email_manager",
    instructions=email_instructions,
    tools=email_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it"
)

In [ ]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [ ]:
sales_manager_instructions = """
You are a Sales Manager at Fun-EduTech SaaS company . Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the email_manager agent. The email_manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the email_manager — never more than one.
"""


# sales_manager = Agent(
#     name="sales_manager",
#     instructions=sales_manager_instructions,
#     tools=tools,
#     handoffs=handoffs,
#     model="gpt-4o-mini")

# message = "Send out a cold sales email addressed to Dear CEO from Priya"

# with trace("Automated SDR"):
#     result = await Runner.run(sales_manager, message)

Implement GuardRails

In [ ]:
# Check if the message output has a name

class NameCheckOutput(BaseModel):
    is_name_in_message:bool
    name:str

guardrail_agent = Agent(
    name="name_check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)    

In [ ]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

In [ ]:
careful_sales_manager = Agent(
    name="sales_manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name]
    )

message = "Send out a cold sales email addressed to Dear CEO from Business Development Manager"

with trace("Protected Automated SDR"):
    
        result = await Runner.run(careful_sales_manager, message)
   

In [ ]:
# message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

# with trace("Protected Automated SDR"):
#     result = await Runner.run(careful_sales_manager, message)